In [1]:
import numpy as np

def shortrad(opt, qz, qs, qcan, Sd, Sq, w, h, aW, aWe, aG, aGe, aR, FGS, FWW, FGW, FWG, FWS, nW, nG, nR):
    """
    Compute shortwave radiation budget for street canyon using Python
    for each subdivided type of surface, using individual albedo
    of its own type and effective albedo of other surfaces.

    Parameters:
    opt: int
        Option for the calculation method (1 for Kusaka, 2 for Masson).
    qz: float
        Zenith angle of the sun.
    qs: float
        Solar azimuth angle.
    qcan: float
        Canyon azimuth angle.
    Sd, Sq: float
        Direct and diffuse solar radiation.
    w, h: float
        Canyon width and height.
    aW, aG, aR: arrays
        Albedos for wall, ground, and roof surfaces respectively.
    FGS, FWW, FGW, FWG, FWS: floats
        View factors.
    nW, nG, nR: int
        Number of wall, ground, and roof subdivisions.

    Returns:
    SW, SG, SR: arrays
        Shortwave radiation for walls, ground, and roof surfaces.
    """

    FG = FGS
    FW = FWS

    # Shadow length
    qn = abs(qcan - qs)
    lsh = h * np.tan(qz) * np.sin(qn)
    lsh = min(lsh, w)

    SR = np.zeros(nR)
    SW1 = np.zeros(nW)
    SW2 = np.zeros(nW)
    SG1 = np.zeros(nG)
    SG2 = np.zeros(nG)

    if opt == 1:  # Kusaka method
        for j1 in range(nR):
            SR[j1] = Sd * (1 - aR[j1]) + Sq * (1 - aR[j1])

        for j2 in range(nW):
            SW1[j2] = Sd * lsh * (1 - aW[j2]) / (2 * h) + Sq * FWS * (1 - aW[j2])
            SW2[j2] = (Sd * (w - lsh) * aGe * FWG * (1 - aW[j2]) / w
                       + Sq * FWG * (1 - aW[j2])
                       + Sd * lsh * aW[j2] * FWW * (1 - aW[j2]) / (2 * h)
                       + Sq * FWS * aW[j2] * FWW * (1 - aW[j2]))

        for j3 in range(nG):
            SG1[j3] = Sd * (w - lsh) * (1 - aG[j3]) / w + Sq * FGS * (1 - aG[j3])
            SG2[j3] = (Sd * lsh * aWe * FGW * (1 - aG[j3]) / (2 * h)
                       + Sq * FWS * aWe * FGW * (1 - aG[j3]))

        SW = SW1 + SW2
        SG = SG1 + SG2

    elif opt == 2:  # Masson method
        SR = Sd * (1 - aR) + Sq * (1 - aR)
        q0 = np.arctan(w / h)

        if qz > q0:
            SWd = 0.5 * w * Sd / h
            SGd = 0
        else:
            SWd = 0.5 * np.tan(qz) * np.sin(qn) * Sd
            SGd = (1 - lsh / (w * np.sin(qn))) * np.sin(qn) * Sd

        f = np.ones(nG)  # Assuming 'f' is a coefficient or scalar array
        RG0 = np.sum(f * (aG * SGd + aG * Sq), axis=0)
        RW0 = aW * SWd + aW * Sq

        x1 = (1 - FG) * FW * f * aG * aW
        x2 = (1 - FG) * f * aG * (RW0 + FW * aW * RG0)
        X1 = 1 - (1 - 2 * FW) * aW + np.sum(x1, axis=0)
        X2 = np.sum(x2, axis=0)

        MG = (RG0 + X2) / X1
        MW = (RW0 + FW * aW * RG0) / X1

        SG = (SGd * (1 - f * aG) + Sq * (1 - f * aG)
              + (1 - f * aG) * (1 - FG) * MW)
        SW = (SWd * (1 - aW) + Sq * (1 - aW)
              + (1 - aW) * (1 - 2 * FW) * MW
              + (1 - aW) * FW * MG)

    return SW, SG, SR
